# <font color="#418FDE" size="6.5" uppercase>**Centroid Clustering**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply K-means and Mini-Batch K-means to unlabeled data. 
- Use inertia and silhouette analysis to compare cluster counts. 
- Fit Affinity Propagation and Mean Shift with key parameter controls. 


## **1. K Means Basics**

### **1.1. K Means Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_01_01.jpg?v=1784020320" width="250">



>* Groups unlabeled data around chosen centroids
>* Reveals similarity patterns for later interpretation

>* K-means finds groups without labels
>* Cluster count strongly shapes results

>* Full K-means favors stable, precise clusters
>* Mini-Batch K-means trades precision for speed



### **1.2. Minimizing Cluster Variance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_01_02.jpg?v=1784020322" width="250">



>* K-means forms groups around nearby centroids
>* Lower variance means more compact, similar clusters

>* Low variance means points near centroids
>* Compact clusters support clearer interpretation

>* Scale features before applying K-means
>* Interpret clusters for real-world meaning



### **1.3. K Means Fitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_01_03.jpg?v=1784020324" width="250">



>* Choose clusters and initialize centroids
>* Iteratively assign points and update centers

>* Starting centroids can change final clusters
>* Repeated smart starts improve clustering reliability

>* Scale features so distances are meaningful
>* Use labels and centroids to summarize groups



In [ ]:
#@title Python Code - K Means Fitting

# This example fits K-means to unlabeled points.
# Scaling makes distance comparisons fair for clustering.
# The plot shows fitted labels and centroids.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Create a small unlabeled dataset with three natural groups.
features, hidden_groups = make_blobs(
    n_samples=180,
    centers=3,
    cluster_std=1.2,
    random_state=42,
)

# Validate the dataset shape before fitting the model.
if features.shape != (180, 2):
    raise ValueError("Expected 180 rows and 2 features.")

# Scale features so both axes influence distances similarly.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Fit K-means with a chosen number of clusters.
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(scaled_features)

# Convert centroids back to the original feature scale.
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

# Print concise results that summarize the fitted clustering.
print("scikit-learn version:", sklearn.__version__)
print("Chosen clusters:", kmeans.n_clusters)
print("Inertia after fitting:", round(kmeans.inertia_, 2))
print("First 10 fitted labels:", cluster_labels[:10].tolist())

# Plot the fitted labels and learned centroids.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=cluster_labels,
    cmap="viridis",
    s=35,
)

# Mark centroids as large red X symbols.
ax.scatter(
    centroids[:, 0],
    centroids[:, 1],
    c="red",
    marker="X",
    s=220,
    label="Centroids",
)

# Label the plot for beginner interpretation.
ax.set_title("K-means fitting on unlabeled data")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()

plt.show()



## **2. K Means Evaluation**

### **2.1. K Means Initialization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_02_01.jpg?v=1784020318" width="250">



>* Initial centroids strongly shape K-means results
>* Poor starts can create misleading clusters

>* Poor starts can inflate inertia scores
>* Awkward clusters can lower silhouette scores

>* Spread centroids for more stable clusters
>* Interpret inertia and silhouette with context



In [ ]:
#@title Python Code - K Means Initialization

# Compare K-means initialization choices on generated clusters.
# Initialization can change inertia and silhouette scores.
# Better starts make cluster-count evaluation more trustworthy.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

# Generate a small unlabeled dataset with three natural groups.
features, true_groups = make_blobs(
    n_samples=450,
    centers=3,
    cluster_std=1.25,
    random_state=42,
)

# Validate the dataset shape before fitting the model.
if features.shape != (450, 2):
    raise ValueError("Expected 450 rows and 2 features.")

# Fit K-means once with random initialization.
random_model = KMeans(
    n_clusters=3,
    init="random",
    n_init=1,
    random_state=42,
)

random_labels = random_model.fit_predict(features)
random_silhouette = silhouette_score(features, random_labels)

# Fit K-means once with the smarter k-means++ initialization.
plus_model = KMeans(
    n_clusters=3,
    init="k-means++",
    n_init=1,
    random_state=42,
)

plus_labels = plus_model.fit_predict(features)
plus_silhouette = silhouette_score(features, plus_labels)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Random init inertia: {random_model.inertia_:.1f}")
print(f"Random init silhouette: {random_silhouette:.3f}")
print(f"k-means++ inertia: {plus_model.inertia_:.1f}")
print(f"k-means++ silhouette: {plus_silhouette:.3f}")

# Plot the k-means++ result used for the stronger evaluation.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=plus_labels,
    cmap="viridis",
    s=28,
)

ax.scatter(
    plus_model.cluster_centers_[:, 0],
    plus_model.cluster_centers_[:, 1],
    c="red",
    marker="X",
    s=180,
    label="Final centroids",
)

ax.set_title("K-means++ initialization with three clusters")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



### **2.2. Multiple Initializations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_02_02.jpg?v=1784020314" width="250">



>* Starting centroids strongly affect K-means results
>* Single runs can mislead inertia evaluation

>* Repeat K-means with different starting centroids
>* Compare cluster counts using more reliable inertia

>* Repeated starts stabilize inertia and silhouette comparisons
>* Balance scores with practical cluster meaning



In [ ]:
#@title Python Code - Multiple Initializations

# This example compares K-means initialization choices.
# Repeated starts make inertia comparisons more reliable.
# The plot shows single versus multiple starts.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

# A fixed seed makes the generated clusters reproducible.
features, true_groups = make_blobs(
    n_samples=450,
    centers=4,
    cluster_std=1.25,
    random_state=42,
)

# This check keeps the example clear and safe.
if features.shape != (450, 2):
    raise ValueError("Expected 450 rows and 2 features.")

cluster_counts = [2, 3, 4, 5, 6]
single_start_inertia = []
multiple_start_inertia = []
multiple_start_silhouette = []

# Each k is evaluated once, then with repeated starts.
for k in cluster_counts:
    single_model = KMeans(n_clusters=k, n_init=1, random_state=42)
    single_labels = single_model.fit_predict(features)

    repeated_model = KMeans(n_clusters=k, n_init=20, random_state=42)
    repeated_labels = repeated_model.fit_predict(features)

    single_start_inertia.append(single_model.inertia_)
    multiple_start_inertia.append(repeated_model.inertia_)
    multiple_start_silhouette.append(silhouette_score(features, repeated_labels))

print("scikit-learn version:", sklearn.__version__)
print("Best k by repeated-start silhouette:", cluster_counts[int(np.argmax(multiple_start_silhouette))])
print("Repeated starts lower inertia for k=4:", round(single_start_inertia[2] - multiple_start_inertia[2], 1))

# The elbow curve is steadier after multiple initializations.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cluster_counts, single_start_inertia, marker="o", label="n_init=1")
ax.plot(cluster_counts, multiple_start_inertia, marker="o", label="n_init=20")
ax.set_title("K-means inertia with different initialization counts")
ax.set_xlabel("Number of clusters k")
ax.set_ylabel("Inertia, lower is better")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **2.3. Mini Batch K Means**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_02_03.jpg?v=1784020316" width="250">



>* Speeds clustering with small random mini batches
>* Supports faster inertia and silhouette comparisons

>* Inertia shows cluster tightness across k values
>* Look for stable elbow patterns, not tiny differences

>* Use silhouette scores to compare cluster quality
>* Balance metrics with stability and practical needs



In [ ]:
#@title Python Code - Mini Batch K Means

# Compare Mini Batch K Means cluster counts.
# Inertia and silhouette show different evidence.
# The plot highlights a reasonable candidate k.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.cluster import MiniBatchKMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Generate unlabeled data with visible cluster structure.
features, true_groups = make_blobs(
    n_samples=900,
    centers=4,
    cluster_std=1.25,
    random_state=42,
)

# Scale features so distances treat both columns fairly.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the data shape before clustering.
if scaled_features.shape != (900, 2):
    raise ValueError("Expected 900 rows and 2 features.")

cluster_counts = [2, 3, 4, 5, 6, 7]
inertia_values = []
silhouette_values = []

# Fit one Mini Batch K Means model for each k.
for cluster_count in cluster_counts:
    model = MiniBatchKMeans(
        n_clusters=cluster_count,
        batch_size=64,
        n_init=10,
        random_state=42,
    )

    labels = model.fit_predict(scaled_features)
    inertia_values.append(model.inertia_)
    silhouette_values.append(silhouette_score(scaled_features, labels))

best_index = int(np.argmax(silhouette_values))
best_k = cluster_counts[best_index]

print(f"scikit-learn version: {sklearn_version}")
print(f"Best silhouette score occurs at k={best_k}.")
print("Inertia falls as k increases, so look for diminishing returns.")

# Plot both metrics after scaling them to comparable ranges.
inertia_array = np.array(inertia_values)
silhouette_array = np.array(silhouette_values)
scaled_inertia = inertia_array / inertia_array.max()
scaled_silhouette = silhouette_array / silhouette_array.max()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cluster_counts, scaled_inertia, marker="o", label="Scaled inertia")
ax.plot(cluster_counts, scaled_silhouette, marker="s", label="Scaled silhouette")
ax.set_title("Mini Batch K Means Evaluation")
ax.set_xlabel("Number of clusters k")
ax.set_ylabel("Scaled metric value")
ax.legend()
plt.show()



## **3. Mode Clustering**

### **3.1. Silhouette Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_03_01.jpg?v=1784020308" width="250">



>* Measure cluster cohesion and separation
>* Compare mode clustering parameter choices

>* Tune preference to balance exemplar counts
>* Use silhouettes to confirm meaningful clusters

>* Compare Mean Shift bandwidths with silhouette scores
>* Balance separation with domain judgment



In [ ]:
#@title Python Code - Silhouette Scores

# Compare mode clustering settings with silhouette scores.
# Affinity Propagation and Mean Shift choose clusters indirectly.
# Higher scores suggest clearer cluster separation.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.cluster import AffinityPropagation
from sklearn.cluster import MeanShift
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Create a small unlabeled dataset with visible group structure.
features, true_groups = make_blobs(
    n_samples=300, centers=4, cluster_std=0.75, random_state=42
)

# Standardizing keeps distance-based clustering on a fair scale.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the data shape before fitting clustering models.
if scaled_features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 features.")

print(f"scikit-learn version: {sklearn_version}")
print("Method and setting | clusters | silhouette")

# Try a few parameter settings for two mode clustering methods.
results = []
settings = [
    ("Affinity Propagation", -80, None),
    ("Affinity Propagation", -40, None),
    ("Mean Shift", None, 0.8),
    ("Mean Shift", None, 1.2),
]

for method_name, preference, bandwidth in settings:
    if method_name == "Affinity Propagation":
        model = AffinityPropagation(
            preference=preference, damping=0.9, random_state=42
        )
        setting_label = f"preference={preference}"
    else:
        model = MeanShift(bandwidth=bandwidth, bin_seeding=True)
        setting_label = f"bandwidth={bandwidth}"

    labels = model.fit_predict(scaled_features)
    cluster_count = len(np.unique(labels))

    if 1 < cluster_count < len(labels):
        score = silhouette_score(scaled_features, labels)
    else:
        score = np.nan

    results.append((method_name, setting_label, cluster_count, score, labels))
    print(f"{method_name}, {setting_label} | {cluster_count} | {score:.3f}")

# Select the valid setting with the highest silhouette score.
valid_results = [row for row in results if not np.isnan(row[3])]
best_result = max(valid_results, key=lambda row: row[3])
best_method, best_setting, best_count, best_score, best_labels = best_result

print(f"Best shown: {best_method}, {best_setting}, score={best_score:.3f}")

# Plot the best clustering result for visual confirmation.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(
    scaled_features[:, 0], scaled_features[:, 1], c=best_labels, s=30
)

ax.set_title("Best Mode Clustering by Silhouette Score")
ax.set_xlabel("Standardized feature 1")
ax.set_ylabel("Standardized feature 2")
plt.show()



### **3.2. Affinity Propagation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_03_02.jpg?v=1784020312" width="250">



>* Finds real data points as exemplars
>* Uses message passing to form clusters

>* Preference controls exemplar count and cluster granularity
>* Damping stabilizes updates and supports convergence

>* Scale features for meaningful similarity comparisons
>* Use exemplars for interpretable smaller-data clusters



In [ ]:
#@title Python Code - Affinity Propagation

# This example clusters unlabeled points with exemplars.
# Affinity Propagation chooses real observations as centers.
# Preference controls the number of discovered clusters.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import AffinityPropagation
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Create a small unlabeled dataset with visible groups.
features, true_groups = make_blobs(
    n_samples=180,
    centers=4,
    cluster_std=0.75,
    random_state=42,
)

# Scale features so similarity is not dominated by one axis.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the data shape before fitting the clustering model.
if scaled_features.shape != (180, 2):
    raise ValueError("Expected 180 rows and 2 numeric features.")

# A higher preference usually allows more exemplars to appear.
preference_value = -35
model = AffinityPropagation(
    preference=preference_value,
    damping=0.85,
    random_state=42,
)

# Fit the model and assign every point to an exemplar.
cluster_labels = model.fit_predict(scaled_features)
exemplar_indices = model.cluster_centers_indices_
cluster_count = len(exemplar_indices)

# Silhouette is meaningful only when at least two clusters exist.
if cluster_count > 1:
    silhouette = silhouette_score(scaled_features, cluster_labels)
else:
    silhouette = 0.0

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Preference value: {preference_value}")
print(f"Clusters found: {cluster_count}")
print(f"Silhouette score: {silhouette:.3f}")

# Plot points by cluster and mark real exemplar observations.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=cluster_labels,
    cmap="tab10",
    s=35,
)

# Exemplar markers show the actual data points chosen as centers.
ax.scatter(
    scaled_features[exemplar_indices, 0],
    scaled_features[exemplar_indices, 1],
    c="black",
    marker="X",
    s=180,
    label="Exemplars",
)

ax.set_title("Affinity Propagation Clusters and Exemplars")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend()
plt.show()



### **3.3. Mean Shift**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_A/image_03_03.jpg?v=1784020310" width="250">



>* Finds dense peaks as cluster centers
>* No preset cluster count required

>* Bandwidth controls Mean Shift neighborhood size
>* Smaller finds detail; larger merges clusters

>* Scale features and manage large dataset costs
>* Use bandwidth to find dense patterns



In [ ]:
#@title Python Code - Mean Shift

# Demonstrate Mean Shift clustering on unlabeled points.
# Bandwidth controls how many dense modes appear.
# The plot shows discovered cluster centers clearly.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import MeanShift
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Create small unlabeled data with three dense regions.
points, true_groups = make_blobs(
    n_samples=240,
    centers=3,
    cluster_std=[0.55, 0.75, 0.6],
    random_state=42,
)

# Scale features so distance-based neighborhoods are comparable.
scaler = StandardScaler()
scaled_points = scaler.fit_transform(points)

# Fit Mean Shift with one clear bandwidth setting.
mean_shift = MeanShift(bandwidth=1.0, bin_seeding=True)
cluster_labels = mean_shift.fit_predict(scaled_points)

# Convert centers back to the original feature scale.
scaled_centers = mean_shift.cluster_centers_
centers = scaler.inverse_transform(scaled_centers)

# Validate the fitted labels before summarizing results.
if len(cluster_labels) != len(points):
    raise ValueError("Each point should receive exactly one cluster label.")

cluster_count = len(np.unique(cluster_labels))
print("scikit-learn version:", sklearn.__version__)
print("Mean Shift bandwidth:", mean_shift.bandwidth)
print("Discovered clusters:", cluster_count)

# Plot points by assigned cluster and mark density peaks.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    points[:, 0],
    points[:, 1],
    c=cluster_labels,
    cmap="viridis",
    s=35,
    alpha=0.8,
)

# Cluster centers are the modes found by Mean Shift.
ax.scatter(
    centers[:, 0],
    centers[:, 1],
    c="red",
    marker="X",
    s=180,
    label="Mean Shift centers",
)

ax.set_title("Mean Shift Finds Dense Modes Without Choosing k")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Centroid Clustering**</font>


In this lecture, you learned to:
- Apply K-means and Mini-Batch K-means to unlabeled data. 
- Use inertia and silhouette analysis to compare cluster counts. 
- Fit Affinity Propagation and Mean Shift with key parameter controls. 

In the next Lecture (Lecture B), we will go over 'Spectral Hierarchical'